In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!cp -a /content/drive/MyDrive/Flow_clean /content/Flow_clean

In [ ]:
!cp -a /content/drive/MyDrive/Flow_celeb /content/Flow_celeb

In [10]:
!mkdir -p /content/Flow_clean && tar -xf /content/drive/MyDrive/flow_100.tar -C /content/Flow_clean

In [4]:
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence

In [ ]:
PROCESSED_FLOWS_PATH = "/content/Flow_clean/Flow_40_100"
CELEB_FLOWS_PATH = "/content/Flow_celeb"
LOMO_HOLDOUT_METHOD = "Deepfakes"

FRAMES = 40
HEIGHT = 100
WIDTH = 100
CHANNELS = 2

TRAIN_MAX_ID = 800
VAL_MAX_ID = 900
TEST_MAX_ID = 950

BALANCE_STRATEGY = "downsample"
RANDOM_SEED = 42

# BATCH_SIZE = 16
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 3e-4
CHECKPOINT_PATH = "./models/best_flow_lstm.keras"
RESULTS_PATH = "./results/lomo_results.json"

ORIGINAL_ALIASES = {"original", "originals", "youtube", "real"}
DFD_ALIASES = {"deepfakedetection", "deepfake_detection", "dfd"}


def keras_metrics():
    return [
        "accuracy",
        tf.keras.metrics.AUC(name="auc"),
        tf.keras.metrics.TruePositives(name="tp"),
        tf.keras.metrics.FalsePositives(name="fp"),
        tf.keras.metrics.TrueNegatives(name="tn"),
        tf.keras.metrics.FalseNegatives(name="fn"),
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall"),
    ]

In [ ]:
def _norm_name(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())


def is_original_folder(folder):
    return _norm_name(folder) in ORIGINAL_ALIASES


def is_deepfake_detection(folder):
    return _norm_name(folder) in {_norm_name(a) for a in DFD_ALIASES}


def folders_match(folder, target):
    return _norm_name(folder) == _norm_name(target)


def extract_video_index(filename):
    match = re.match(r"^(\d+)", Path(filename).name)
    return int(match.group(1)) if match else None


def assign_split(vid_idx, keep_eval_tail):
    if vid_idx < TRAIN_MAX_ID:
        return "train"
    if vid_idx < VAL_MAX_ID:
        return "val"
    if vid_idx < TEST_MAX_ID:
        return "test"
    return "eval" if keep_eval_tail else "test"

In [ ]:
def _empty_splits(include_eval=True):
    names = ("train", "val", "test", "eval") if include_eval else ("train", "val", "test")
    return {
        name: {"x": [], "y": [], "method": []}
        for name in names
    }


def _add(splits, target, path, label, method):
    splits[target]["x"].append(path)
    splits[target]["y"].append(label)
    splits[target]["method"].append(method)


def load_lomo_npy_splits(processed_dir, lomo_holdout_method=None):
    include_eval = lomo_holdout_method is not None
    splits = _empty_splits(include_eval=include_eval)
    processed_dir = Path(processed_dir)
    if not processed_dir.is_dir():
        raise FileNotFoundError(f"Не знайдено папку потоків: {processed_dir}")

    folders = sorted(
        p.name for p in processed_dir.iterdir() if p.is_dir() and not p.name.startswith(".")
    )
    print(f"Знайдені методи: {folders}")
    print(f"LOMO hold-out:   {lomo_holdout_method}")

    holdout_seen = False

    for folder in folders:
        folder_path = processed_dir / folder
        files = sorted(p.name for p in folder_path.glob("*.npy"))
        if not files:
            print(f"  [skip] {folder}: немає .npy")
            continue

        label = 0 if is_original_folder(folder) else 1

        if include_eval and folders_match(folder, lomo_holdout_method):
            holdout_seen = True
            for filename in files:
                _add(splits, "eval", str(folder_path / filename), label, folder)
            print(f"  [hold-out -> eval] {folder}: {len(files)} файлів, label={label}")
            continue

        if is_original_folder(folder):
            for idx, filename in enumerate(files):
                vid_idx = extract_video_index(filename)
                if vid_idx is None:
                    raise ValueError(f"має бути числовий id у original ({filename}), але нема")
                target = assign_split(vid_idx, keep_eval_tail=include_eval)
                _add(splits, target, str(folder_path / filename), label, folder)
            print(f"  [original by vid_idx] {folder}: {len(files)} файлів")
            continue

        if is_deepfake_detection(folder):
            for idx, filename in enumerate(files):
                target = assign_split(idx, keep_eval_tail=False)
                _add(splits, target, str(folder_path / filename), label, folder)
            print(f"  [DeepFakeDetection by file index] {folder}: {len(files)} файлів")
            continue

        for filename in files:
            vid_idx = extract_video_index(filename)
            if vid_idx is None:
                raise ValueError(f"має бути числовий id у original ({filename}), але нема")
            target = assign_split(vid_idx, keep_eval_tail=False)
            _add(splits, target, str(folder_path / filename), label, folder)
        print(f"  [fake by vid_id] {folder}: {len(files)} файлів")

    if include_eval and not holdout_seen:
        raise ValueError(f"Hold-out метод '{lomo_holdout_method}' не знайдено серед {folders}")

    for name in splits:
        splits[name]["x"] = np.array(splits[name]["x"])
        splits[name]["y"] = np.array(splits[name]["y"], dtype=np.int32)
        splits[name]["method"] = np.array(splits[name]["method"])

    return splits


def print_split_stats(splits, title):
    print(f"\n=== {title} ===")
    for name in ("train", "val", "test", "eval"):
        y = splits[name]["y"]
        methods = splits[name]["method"]
        n_real = int(np.sum(y == 0))
        n_fake = int(np.sum(y == 1))
        method_counts = Counter(methods.tolist()) if len(methods) else {}
        print(
            f"  {name:5s}: n={len(y):4d}  real={n_real:4d}  fake={n_fake:4d}  "
            f"ratio={n_fake / max(n_real, 1):.2f}  {dict(method_counts)}"
        )

In [ ]:
def _sample_indices(indices, k, rng):
    if k <= len(indices):
        return rng.choice(indices, size=k, replace=False)
    extra = rng.choice(indices, size=k - len(indices), replace=True)
    return np.concatenate([indices, extra])


def _downsample_fakes_stratified(fake_idx, methods, n_keep, rng):
    by_method = defaultdict(list)
    for i in fake_idx:
        by_method[str(methods[i])].append(int(i))

    method_names = sorted(by_method)
    if not method_names:
        return np.array([], dtype=int)

    base, rem = divmod(n_keep, len(method_names))
    chosen = []
    leftover = []

    for m_i, method in enumerate(method_names):
        pool = np.array(by_method[method], dtype=int)
        quota = base + (1 if m_i < rem else 0)
        take = min(quota, len(pool))
        picked = rng.choice(pool, size=take, replace=False)
        chosen.extend(picked.tolist())
        unused = np.setdiff1d(pool, picked, assume_unique=False)
        leftover.extend(unused.tolist())

    shortage = n_keep - len(chosen)
    if shortage > 0 and leftover:
        extra = _sample_indices(np.array(leftover, dtype=int), shortage, rng)
        chosen.extend(extra.tolist())

    return np.array(chosen[:n_keep], dtype=int)


def balance_split(split, strategy="downsample", seed=RANDOM_SEED):
    paths = np.asarray(split["x"])
    labels = np.asarray(split["y"])
    methods = np.asarray(split["method"])

    real_idx = np.where(labels == 0)[0]
    fake_idx = np.where(labels == 1)[0]
    if len(real_idx) == 0 or len(fake_idx) == 0:
        print("  один із класів порожній — балансування пропущено")
        return {"x": paths, "y": labels, "method": methods}

    rng = np.random.default_rng(seed)

    if strategy == "downsample":
        n = min(len(real_idx), len(fake_idx))
        keep_real = _sample_indices(real_idx, n, rng)
        if len(fake_idx) >= n:
            keep_fake = _downsample_fakes_stratified(fake_idx, methods, n, rng)
        else:
            keep_fake = _sample_indices(fake_idx, n, rng)
        keep = np.concatenate([keep_real, keep_fake])
    elif strategy == "oversample":
        n = max(len(real_idx), len(fake_idx))
        keep_real = _sample_indices(real_idx, n, rng)
        keep_fake = _sample_indices(fake_idx, n, rng)
        keep = np.concatenate([keep_real, keep_fake])
    else:
        raise ValueError(f"Невідома стратегія балансування: {strategy}")

    rng.shuffle(keep)
    return {
        "x": paths[keep],
        "y": labels[keep].astype(np.int32),
        "method": methods[keep],
    }


def balance_all_splits(splits, strategy, seed):
    balanced = {}
    for i, name in enumerate(("train", "val", "test", "eval")):
        balanced[name] = balance_split(splits[name], strategy=strategy, seed=seed + i)
    return balanced

In [9]:
def load_flow_sample(path):
    return np.load(path).astype(np.float32)


class FlowSequence(Sequence):
    def __init__(self, npy_paths, labels, batch_size=16, shuffle=False, **kwargs):
        super().__init__(**kwargs)
        self.npy_paths = np.asarray(npy_paths)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.npy_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.npy_paths) / self.batch_size))

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_idx = self.indices[start:end]
        batch_x = np.stack([load_flow_sample(p) for p in self.npy_paths[batch_idx]])
        batch_y = self.labels[batch_idx]
        return batch_x, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

In [ ]:
def predict_split(model, split, batch_size):
    seq = FlowSequence(split["x"], split["y"], batch_size=batch_size, shuffle=False)
    preds = model.predict(seq, verbose=0)
    if isinstance(preds, dict):
        preds = preds["output"]
    elif isinstance(preds, (list, tuple)):
        preds = preds[0]
    probs = np.asarray(preds).reshape(-1)[: len(split["y"])]
    y_true = np.asarray(split["y"], dtype=np.int32)
    return y_true, probs


def summarize_predictions(y_true, y_prob, methods=None, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(np.int32)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    result = {
        "n": int(len(y_true)),
        "n_real": int(np.sum(y_true == 0)),
        "n_fake": int(np.sum(y_true == 1)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "auc": float("nan"),
        "eer": float("nan"),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "threshold": threshold,
        "confusion_matrix": cm.tolist(),
        "per_method": {},
    }

    if len(np.unique(y_true)) > 1:
        result["auc"] = float(roc_auc_score(y_true, y_prob))
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        result["eer"] = float(fpr[np.nanargmin(np.abs(fpr - (1.0 - tpr)))])

    if methods is not None:
        for method in sorted(set(methods.tolist())):
            mask = methods == method
            if mask.sum() == 0:
                continue
            ym = y_true[mask]
            pm = (y_prob[mask] >= threshold).astype(np.int32)
            entry = {
                "n": int(mask.sum()),
                "accuracy": float(accuracy_score(ym, pm)),
            }
            if len(np.unique(ym)) > 1:
                entry["auc"] = float(roc_auc_score(ym, y_prob[mask]))
            result["per_method"][str(method)] = entry

    return result


def print_eval_report(title, metrics):
    print(f"\n========== {title} ==========")
    print(f"n={metrics['n']}  real={metrics['n_real']}  fake={metrics['n_fake']}")
    print(
        f"acc={metrics['accuracy']:.4f}  prec={metrics['precision']:.4f}  "
        f"rec={metrics['recall']:.4f}  f1={metrics['f1']:.4f}  "
        f"auc={metrics['auc']:.4f}  eer={metrics['eer']:.4f}"
    )
    print(f"tp={metrics['tp']}  fp={metrics['fp']}  tn={metrics['tn']}  fn={metrics['fn']}")
    print("confusion [[TN FP] [FN TP]]:", metrics["confusion_matrix"])
    if metrics["per_method"]:
        print("per-method:")
        for method, vals in metrics["per_method"].items():
            extra = f"  auc={vals['auc']:.4f}" if "auc" in vals else ""
            print(f"  {method:20s}  n={vals['n']:4d}  acc={vals['accuracy']:.4f}{extra}")


def choose_threshold(y_true, y_prob):
    best_t = 0.5
    best_f1 = -1.0
    for t in np.linspace(0.05, 0.95, 91):
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)
    return best_t, best_f1


def evaluate_in_domain_and_lomo(model, raw_splits, balanced_splits, batch_size, threshold=0.5):
    reports = {}

    for key, title, split_name in (
        ("in_domain_balanced", "In-Domain (balanced test)", "test"),
        ("lomo_balanced", "LOMO (balanced eval)", "eval"),
        ("in_domain_full", "In-Domain (full unbalanced test)", "test"),
        ("lomo_full", "LOMO (full unbalanced eval)", "eval"),
    ):
        source = balanced_splits if "balanced" in key else raw_splits
        y_true, y_prob = predict_split(model, source[split_name], batch_size)
        metrics = summarize_predictions(y_true, y_prob, methods=source[split_name]["method"], threshold=threshold)
        print_eval_report(title, metrics)
        y_pred = (y_prob >= threshold).astype(int)
        print(classification_report(y_true, y_pred, target_names=["real", "fake"], digits=4, zero_division=0))
        reports[key] = metrics

    return reports


In [ ]:
def history_to_dict(history):
    return {key: [float(v) for v in values] for key, values in history.history.items()}


def save_results(reports, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(reports, f, ensure_ascii=False, indent=2)

In [13]:
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import LSTM, BatchNormalization, Conv2D, Dense, Dropout, GlobalAveragePooling2D, Input, MaxPooling2D, ReLU, TimeDistributed
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import Sequence

In [14]:
CHECKPOINT_PATH = "./models/best_flow_cnn_lstm.keras"
RESULTS_PATH = "./results/lomo_cnn_lstm_results.json"
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
EPOCHS = 40
FINETUNE_EPOCHS = 12
FINETUNE_LR = 1e-4
MIXUP_ALPHA = 0.2
# MIXUP_ALPHA = 0.1
HFLIP_PROB = 0.5

In [ ]:
def conv_block(x, filters, name):
    x = Conv2D(filters, 3, padding="same", kernel_regularizer=l2(WEIGHT_DECAY), name=name + "_conv")(x)
    x = BatchNormalization(name=name + "_bn")(x)
    x = ReLU(name=name + "_relu")(x)
    return x


def build_frame_cnn(height, width, channels):
    frame = Input(shape=(height, width, channels), name="frame")
    x = conv_block(frame, 32, "block1")
    x = MaxPooling2D(2, name="pool1")(x)  # 50x50
    x = conv_block(x, 48, "block2")
    x = MaxPooling2D(2, name="pool2")(x)  # 25x25
    x = conv_block(x, 64, "block3")
    x = MaxPooling2D(2, name="pool3")(x)  # 12x12
    x = conv_block(x, 64, "block4")
    x = MaxPooling2D(2, name="pool4")(x)  # 6x6
    features = GlobalAveragePooling2D(name="gap")(x)
    return Model(frame, features, name="FrameCNN")


def build_cnn_lstm(frames=FRAMES, height=HEIGHT, width=WIDTH, channels=CHANNELS):
    inputs = Input(shape=(frames, height, width, channels), name="flow_input")
    features = TimeDistributed(build_frame_cnn(height, width, channels), name="cnn_features")(inputs)
    x = Dropout(0.35, name="feat_drop")(features)
    x = LSTM(64, return_sequences=True, kernel_regularizer=l2(WEIGHT_DECAY), name="lstm1")(x)
    x = Dropout(0.45, name="drop1")(x)
    x = LSTM(32, kernel_regularizer=l2(WEIGHT_DECAY), name="lstm2")(x)
    x = Dropout(0.45, name="drop2")(x)
    outputs = Dense(1, activation="sigmoid", name="output")(x)
    return Model(inputs, outputs, name="Flow_CNN_LSTM_100")


def compile_cnn_lstm(model, learning_rate=LEARNING_RATE):
    model.compile(
        optimizer=AdamW(learning_rate=learning_rate, weight_decay=WEIGHT_DECAY, clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=keras_metrics(),
    )
    return model


def hflip_flow(flow):
    out = np.flip(flow, axis=2).copy()
    out[..., 0] *= -1.0
    return out


class MixupFlowSequence(Sequence):

    def __init__(self, npy_paths, labels, batch_size=16, shuffle=True, mixup_alpha=0.2, hflip_prob=0.5, seed=RANDOM_SEED, **kwargs):
        super().__init__(**kwargs)
        self.npy_paths = np.asarray(npy_paths)
        self.labels = np.asarray(labels, dtype=np.float32)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.mixup_alpha = mixup_alpha
        self.hflip_prob = hflip_prob
        self.rng = np.random.default_rng(seed)
        self.indices = np.arange(len(self.npy_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.npy_paths) / self.batch_size))

    def _load_one(self, path):
        flow = load_flow_sample(path)
        if self.rng.random() < self.hflip_prob:
            flow = hflip_flow(flow)
        return flow

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = min(start + self.batch_size, len(self.indices))
        batch_idx = self.indices[start:end]
        x = np.stack([self._load_one(p) for p in self.npy_paths[batch_idx]])
        y = self.labels[batch_idx].copy()
        if self.mixup_alpha > 0 and len(x) > 1:
            lam = float(self.rng.beta(self.mixup_alpha, self.mixup_alpha))
            perm = self.rng.permutation(len(x))
            x = lam * x + (1.0 - lam) * x[perm]
            y = lam * y + (1.0 - lam) * y[perm]
        return x.astype(np.float32), y.astype(np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


def make_splits(raw):
    out = {
        "train": balance_split(raw["train"], strategy="oversample", seed=RANDOM_SEED),
        "val": balance_split(raw["val"], strategy="downsample", seed=RANDOM_SEED + 1),
        "test": balance_split(raw["test"], strategy="downsample", seed=RANDOM_SEED + 2),
    }
    if "eval" in raw:
        out["eval"] = balance_split(raw["eval"], strategy="downsample", seed=RANDOM_SEED + 3)
    return out

In [16]:
def make_callbacks(checkpoint_path):
    Path(checkpoint_path).parent.mkdir(parents=True, exist_ok=True)
    return [
        EarlyStopping(monitor="val_loss", mode="min", patience=10, restore_best_weights=True, verbose=1),
        ModelCheckpoint(checkpoint_path, monitor="val_loss", mode="min", save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]


def _fit(model, train_seq, val_seq, epochs, checkpoint_path, initial_epoch=0):
    return model.fit(
        train_seq,
        steps_per_epoch=len(train_seq),
        validation_data=val_seq,
        validation_steps=len(val_seq),
        epochs=epochs,
        initial_epoch=initial_epoch,
        callbacks=make_callbacks(checkpoint_path),
        verbose=1,
    )


def train_cnn_lstm(model, balanced, batch_size, epochs, checkpoint_path):
    train_seq = MixupFlowSequence(
        balanced["train"]["x"],
        balanced["train"]["y"],
        batch_size=batch_size,
        shuffle=True,
        mixup_alpha=MIXUP_ALPHA,
        hflip_prob=HFLIP_PROB,
        seed=RANDOM_SEED,
    )
    val_seq = FlowSequence(balanced["val"]["x"], balanced["val"]["y"], batch_size=batch_size, shuffle=False)
    print(f"steps_per_epoch={len(train_seq)}  validation_steps={len(val_seq)}")
    print(f"train: hflip p={HFLIP_PROB}, mixup alpha={MIXUP_ALPHA}")

    hist1 = _fit(model, train_seq, val_seq, epochs, checkpoint_path)

    if Path(checkpoint_path).exists():
        model.load_weights(checkpoint_path)
    compile_cnn_lstm(model, learning_rate=FINETUNE_LR)
    print(f"\nFine-tune {FINETUNE_EPOCHS} епох, lr={FINETUNE_LR}, без mixup")
    train_ft = MixupFlowSequence(
        balanced["train"]["x"],
        balanced["train"]["y"],
        batch_size=batch_size,
        shuffle=True,
        mixup_alpha=0.0,
        hflip_prob=HFLIP_PROB,
        seed=RANDOM_SEED + 1,
    )
    hist2 = _fit(model, train_ft, val_seq, epochs + FINETUNE_EPOCHS, checkpoint_path, initial_epoch=epochs)

    merged = {}
    for key in hist1.history:
        merged[key] = list(hist1.history[key]) + list(hist2.history.get(key, []))
    for key in hist2.history:
        if key not in merged:
            merged[key] = list(hist2.history[key])

    class _H:
        pass

    out = _H()
    out.history = merged
    return out

In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

LOMO_HOLDOUT_METHOD = "Deepfakes"
CHECKPOINT_PATH = f"./models/Flow_100.32B-A_{LOMO_HOLDOUT_METHOD}.keras"
HIST_PATH = f"./models/Flow_100.32B-A_{LOMO_HOLDOUT_METHOD}_hist.npy"
REPORT_PATH = f"./models/Flow_100.32B-A_{LOMO_HOLDOUT_METHOD}_report.npy"

raw = load_lomo_npy_splits(PROCESSED_FLOWS_PATH, LOMO_HOLDOUT_METHOD)
# print_split_stats(raw, "До балансування")

balanced = make_splits(raw)
# print_split_stats(balanced, "Після балансування (train=oversample, val/test/eval=downsample)")

model = build_cnn_lstm()
compile_cnn_lstm(model)
model.summary()
model.get_layer("cnn_features").layer.summary()

In [ ]:
history = train_cnn_lstm(model, balanced, BATCH_SIZE, EPOCHS, CHECKPOINT_PATH)

In [ ]:
if Path(CHECKPOINT_PATH).exists():
    model.load_weights(CHECKPOINT_PATH)
    print(f"Завантажено ваги з {CHECKPOINT_PATH}")

y_val, p_val = predict_split(model, balanced["val"], BATCH_SIZE)
thr, val_f1 = choose_threshold(y_val, p_val)
print(f"\nПоріг з val (max F1={val_f1:.4f}): {thr:.3f}  (замість 0.5)")
print_eval_report("Val @ 0.5", summarize_predictions(y_val, p_val, methods=balanced["val"]["method"], threshold=0.5))
print_eval_report("Val @ tuned", summarize_predictions(y_val, p_val, methods=balanced["val"]["method"], threshold=thr))

reports = evaluate_in_domain_and_lomo(model, raw, balanced, batch_size=BATCH_SIZE, threshold=thr)
reports["history"] = history_to_dict(history)
reports["val_threshold"] = thr
reports["config"] = {
    "model": "cnn_lstm_gap",
    "processed_flows_path": PROCESSED_FLOWS_PATH,
    "holdout_method": LOMO_HOLDOUT_METHOD,
    "train_balance": "oversample",
    "eval_balance": "downsample",
    "frames": FRAMES,
    "input_shape": [FRAMES, HEIGHT, WIDTH, CHANNELS],
    "checkpoint_monitor": "val_loss",
    "weight_decay": WEIGHT_DECAY,
    "learning_rate": LEARNING_RATE,
    "finetune_lr": FINETUNE_LR,
    "mixup_alpha": MIXUP_ALPHA,
    "hflip_prob": HFLIP_PROB,
    "eval_threshold": "val_max_f1",
    "thresholds": {
        "train_max_id": TRAIN_MAX_ID,
        "val_max_id": VAL_MAX_ID,
        "test_max_id": TEST_MAX_ID,
    },
}

In [ ]:
# model.save("/content/drive/MyDrive/models/FLOW_CNN_LSTM_simple.keras")
np.save(HIST_PATH, history.history)
np.save(REPORT_PATH, reports)
# np.save("/content/models/FLOW_CNN_LSTM_simple_full.npy", histoty)